In [9]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from google.cloud import bigquery
from witwidget.notebook.visualization import WitWidget, WitConfigBuilder

In [2]:
query="""
SELECT
  weight_pounds,
  is_male,
  mother_age,
  plurality,
  gestation_weeks
FROM
  publicdata.samples.natality
WHERE year > 2000
LIMIT 10000
"""
df = bigquery.Client().query(query).to_dataframe()
df.head()

,weight_pounds,is_male,mother_age,plurality,gestation_weeks
0,7.625790,False,23,1,39
1,8.000575,True,24,1,41
2,3.062221,True,16,2,30
3,7.813183,False,32,1,41
4,7.187070,False,35,1,44


In [3]:
df.describe()

,weight_pounds,mother_age,plurality,gestation_weeks
count,9988.000000,10000.0,10000.0,9947.0
mean,7.245341,27.4575,1.035,38.605409
std,1.341440,6.168241,0.198945,2.638945
min,0.661387,13.0,1.0,17.0
25%,6.569775,23.0,1.0,38.0
50%,7.329268,27.0,1.0,39.0
75%,8.062305,32.0,1.0,40.0
max,12.125424,50.0,5.0,47.0


In [4]:
df = df.dropna()
df = shuffle(df, random_state=2)

In [5]:
labels = df['weight_pounds']
data = df.drop(columns=['weight_pounds'])
data['is_male'] = data['is_male'].astype(int)

#### Prepare data and model

In [22]:
x,y = data,labels
x_train,x_test,y_train,y_test = train_test_split(x,y)
x_train = x_train.to_numpy(dtype='float32')
y_train = y_train.to_numpy(dtype='int32')
x_test = x_test.to_numpy(dtype='float32')
y_test = y_test.to_numpy(dtype='int32')

In [18]:
model = Sequential()
model.add(Input(shape=(len(x_train[0]),)))  # Specify the input shape
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='relu'))

In [19]:
model.compile(optimizer=tf.keras.optimizers.RMSprop(),
              loss=tf.keras.losses.MeanSquaredError(),
              metrics=['mae', 'mse'])

In [20]:
model.fit(x_train, y_train, epochs=10, validation_split=0.1)

Epoch 1/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.6612 - mae: 1.0057 - mse: 1.6612 - val_loss: 1.7261 - val_mae: 1.0497 - val_mse: 1.7261
Epoch 2/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.4186 - mae: 0.9282 - mse: 1.4186 - val_loss: 1.4066 - val_mae: 0.9043 - val_mse: 1.4066
Epoch 3/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.3712 - mae: 0.9181 - mse: 1.3712 - val_loss: 1.3997 - val_mae: 0.9023 - val_mse: 1.3997
Epoch 4/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.3920 - mae: 0.9109 - mse: 1.3920 - val_loss: 1.6244 - val_mae: 1.0057 - val_mse: 1.6244
Epoch 5/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.4054 - mae: 0.9228 - mse: 1.4054 - val_loss: 1.4805 - val_mae: 0.9491 - val_mse: 1.4805
Epoch 6/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.3810 - mae: 0.9122 - mse: 1.3810 - val_loss: 1.3773 - val_mae: 0.8985 - val_mse: 1.3773
Epoch 7/10
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.3765 - mae: 0.9143 - mse: 1.3765 - val

In [23]:
num_examples = 10
predictions = model.predict(x_test[:num_examples])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step


In [24]:
for i in range(num_examples):
    print('Predicted val: ', predictions[i][0])
    print('Actual val: ',y_test[i])
    print()

Predicted val:  6.9320745
Actual val:  6

Predicted val:  7.2892675
Actual val:  7

Predicted val:  7.1084027
Actual val:  10

Predicted val:  6.8220363
Actual val:  7

Predicted val:  6.6826186
Actual val:  7

Predicted val:  6.705278
Actual val:  8

Predicted val:  6.7703295
Actual val:  7

Predicted val:  6.470807
Actual val:  8

Predicted val:  7.165167
Actual val:  7

Predicted val:  6.6588516
Actual val:  6

